In [12]:
import numpy as np
import pandas as pd
import os
import cv2
from PIL import Image
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm

In [4]:
#路徑處理

train_dir = "xray_data/train"
test_dir  = "xray_data/test"

print("Train exists:", os.path.exists(train_dir))
print("Test exists:", os.path.exists(test_dir))

print("Train classes:", os.listdir(train_dir)[:15])
print("Test samples:", os.listdir(test_dir)[:10])

Train exists: True
Test exists: True
Train classes: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']
Test samples: ['0.jpg', '1.jpg', '10.jpg', '100.jpg', '1000.jpg', '1001.jpg', '1002.jpg', '1003.jpg', '1004.jpg', '1005.jpg']


In [5]:
# 影像預處理方法

IMG_SIZE = 224

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# =========================
# Image Processing Classes
# =========================

class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        img = np.array(img.convert("L"))

        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )

        img = clahe.apply(img)
        img = Image.fromarray(img).convert("RGB")
        return img


class SharpenTransform:
    def __call__(self, img):
        img = np.array(img)

        kernel = np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ])

        sharpened = cv2.filter2D(img, -1, kernel)
        sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)

        return Image.fromarray(sharpened)


class GammaCorrection:
    def __init__(self, gamma=0.8):
        self.gamma = gamma

    def __call__(self, img):
        img = np.array(img).astype(np.float32) / 255.0
        img = np.power(img, self.gamma)
        img = np.clip(img * 255, 0, 255).astype(np.uint8)

        return Image.fromarray(img).convert("RGB")


class DenoiseTransform:
    def __call__(self, img):
        img = np.array(img.convert("L"))
        denoised = cv2.GaussianBlur(img, (3, 3), 0)

        return Image.fromarray(denoised).convert("RGB")

class HybridXrayEnhancement:
    def __init__(self, gamma=0.9, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.gamma = gamma
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        # 1. Convert to grayscale
        img = np.array(img.convert("L"))

        # 2. Light denoising
        img = cv2.GaussianBlur(img, (3, 3), 0)

        # 3. CLAHE local contrast enhancement
        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )
        img = clahe.apply(img)

        # 4. Gamma correction
        img = img.astype(np.float32) / 255.0
        img = np.power(img, self.gamma)
        img = np.clip(img * 255, 0, 255).astype(np.uint8)

        # 5. Light sharpening
        kernel = np.array([
            [0, -0.5, 0],
            [-0.5, 3.0, -0.5],
            [0, -0.5, 0]
        ])
        img = cv2.filter2D(img, -1, kernel)
        img = np.clip(img, 0, 255).astype(np.uint8)

        # 6. Convert back to RGB for pretrained models
        return Image.fromarray(img).convert("RGB")


# =========================
# Select One Experiment
# =========================

# 五個擇一，把要跑的那個取消註解

EXPERIMENT = "baseline"
# EXPERIMENT = "clahe"
# EXPERIMENT = "clahe_sharpen"
# EXPERIMENT = "gamma"
# EXPERIMENT = "denoise_clahe"
# EXPERIMENT = "hybrid"

# 是否加入資料增強
# USE_AUGMENTATION = True
USE_AUGMENTATION = False


# =========================
# Build preprocessing pipeline
# =========================

def get_preprocess_ops(experiment):

#  回傳主要影像處理方法
    
    if experiment == "baseline":
        return [
            transforms.Grayscale(num_output_channels=3)
        ]

    elif experiment == "clahe":
        return [
            CLAHETransform()
        ]

    elif experiment == "clahe_sharpen":
        return [
            CLAHETransform(),
            SharpenTransform()
        ]

    elif experiment == "gamma":
        return [
            transforms.Grayscale(num_output_channels=3),
            GammaCorrection(gamma=0.8)
        ]

    elif experiment == "denoise_clahe":
        return [
            DenoiseTransform(),
            CLAHETransform()
        ]

    elif experiment == "hybrid":
        return [
            HybridXrayEnhancement(
                gamma=0.9,
                clip_limit=2.0,
                tile_grid_size=(8, 8)
            )
        ]

    else:
        raise ValueError(f"Unknown experiment: {experiment}")


preprocess_ops = get_preprocess_ops(EXPERIMENT)


# Train transform
if USE_AUGMENTATION:
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=7),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        *preprocess_ops,
        transforms.ToTensor(),
        normalize
    ])
else:
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        *preprocess_ops,
        transforms.ToTensor(),
        normalize
    ])


# Validation / Test transform
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    *preprocess_ops,
    transforms.ToTensor(),
    normalize
])

print("Current experiment:", EXPERIMENT)
print("Use augmentation:", USE_AUGMENTATION)

Current experiment: baseline
Use augmentation: False


In [6]:
# dataset的建立用來訓練讀檔的

base_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=None
)

targets = np.array(base_dataset.targets)
indices = np.arange(len(base_dataset))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=targets
)

train_full = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

val_full = datasets.ImageFolder(
    root=train_dir,
    transform=val_test_transform
)

train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(val_full, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Total images:", len(base_dataset))
print("Train images:", len(train_dataset))
print("Val images:", len(val_dataset))
print("Classes:", base_dataset.classes)

Total images: 22921
Train images: 18336
Val images: 4585
Classes: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


In [7]:
# 1. Device Configuration (Utilize GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Model Definition (ResNet50)
# Pretrained weights provide a robust feature extraction foundation for chest X-rays
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Modify the final fully connected layer to output the correct number of classes (15)
num_classes = len(base_dataset.classes) 
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# 3. Loss Function & Optimization Configuration
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print(f"Model successfully loaded. Class count: {num_classes}")

Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\kaich/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:09<00:00, 10.6MB/s]


Model successfully loaded. Class count: 15


In [8]:
# 4. Training and Validation Process
num_epochs = 10
best_val_acc = 0.0

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Training")
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
        train_bar.set_postfix(loss=loss.item())
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = correct_train / total_train
    
    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_loader.dataset)
    epoch_val_acc = correct_val / total_val
    
    # Step the learning rate scheduler
    scheduler.step()
    
    print(f"Epoch [{epoch+1}/{num_epochs}] Results:")
    print(f"  Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}%")
    print(f"  Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")
    
    # Checkpoint evaluation & saving the best model weights
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), "best_resnet50.pth")
        print("  => New best validation accuracy! Saved model weights.")

print(f"\nTraining completed. Highest Validation Accuracy achieved: {best_val_acc*100:.2f}%")

Epoch [1/10] Training: 100%|██████████| 573/573 [02:03<00:00,  4.65it/s, loss=1.92]


Epoch [1/10] Results:
  Train Loss: 2.2401 | Train Acc: 25.66%
  Val Loss: 2.0721 | Val Acc: 31.43%
  => New best validation accuracy! Saved model weights.


Epoch [2/10] Training: 100%|██████████| 573/573 [02:02<00:00,  4.68it/s, loss=1.5] 


Epoch [2/10] Results:
  Train Loss: 1.8953 | Train Acc: 37.45%
  Val Loss: 2.0141 | Val Acc: 33.52%
  => New best validation accuracy! Saved model weights.


Epoch [3/10] Training: 100%|██████████| 573/573 [02:05<00:00,  4.56it/s, loss=1.32] 


...

Epoch [8/10] Results:
  Train Loss: 0.0152 | Train Acc: 99.86%
  Val Loss: 3.1539 | Val Acc: 32.98%


Epoch [9/10] Training: 100%|██████████| 573/573 [02:04<00:00,  4.60it/s, loss=0.0127] 


Epoch [9/10] Results:
  Train Loss: 0.0095 | Train Acc: 99.96%
  Val Loss: 3.1954 | Val Acc: 33.02%


Epoch [10/10] Training: 100%|██████████| 573/573 [02:01<00:00,  4.73it/s, loss=0.00756]


Epoch [10/10] Results:
  Train Loss: 0.0078 | Train Acc: 99.97%
  Val Loss: 3.1894 | Val Acc: 32.43%

Training completed. Highest Validation Accuracy achieved: 33.81%


In [ ]:
# 5. Custom Test Dataset Loader Implementation
class XRayTestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        files = os.listdir(test_dir)
        
        # Sort files numerically (0.jpg, 1.jpg...) to maintain sequential structural consistency
        try:
            self.filenames = sorted(files, key=lambda x: int(os.path.splitext(x)[0]))
        except ValueError:
            self.filenames = sorted(files)

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.test_dir, filename)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, filename

# Instantiate Test DataLoader
test_dataset = XRayTestDataset(test_dir=test_dir, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Load the best-saved model checkpoint parameters
model.load_state_dict(torch.load("best_resnet50.pth"))
model.eval()

submission_data = []
class_names = base_dataset.classes # Pull string mapping directly from ImageFolder classes

# --- Inference / Prediction Generation Phase ---
print("Starting testing inference on test set files...")
with torch.no_grad():
    for images, filenames in tqdm(test_loader, desc="Testing Inference"):
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        for i in range(len(filenames)):
            filename = filenames[i]
            pred_class_idx = predicted[i].item()
            pred_class_name = class_names[pred_class_idx]
            
            # Formulate tracking index match format
            row_id = len(submission_data)
            submission_data.append({
                "id": row_id,
                "filename": filename,
                "label": pred_class_name
            })

# 6. Structuring into DataFrame and Exporting Target File
submission_df = pd.DataFrame(submission_data)
submission_df.to_csv("submission.csv", index=False)
print("Submission file generation successfully concluded. Output saved as 'submission.csv'!")

Starting testing inference on test set files...


Testing Inference:   0%|          | 0/157 [00:05<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 144304, 169352) exited unexpectedly